# LangGraph 04 · 时间旅行、子图与容错

这一课把「图」当**数据**来玩：checkpointer 把每一步执行存成快照，于是可以回退重跑
（**时间旅行**）；一张编译好的图可以当成另一个图的节点（**子图**）；在此基础上，
两篇官方补充补齐了工程化的两块拼图 —— 子图持久化三档，以及节点的容错与测试。

四个主题，一张表先扫一遍：

| 主题 | 一句话 | 来源脚本 |
|---|---|---|
| 时间旅行 | `MemorySaver` 存了每一步快照，可 `update_state` 回退、`invoke(None)` 重跑 | `08_时间旅行` |
| 子图 | 已编译的图当节点用，同名 state 键自动透传 | `09_子图` |
| 子图持久化 | 子图 `checkpointer` 三档：per-invocation / per-thread / stateless | `14_子图持久化_官方补充` |
| 容错与测试 | `RetryPolicy` / 节点超时 / 三种 pytest 粒度 | `11_容错与测试_官方补充` |

> **本 notebook 由 `Agent/01_langgraph/` 下 6 个脚本合并而成**：
> `08_时间旅行.py` + `08_时间旅行_jxsd.py`、`09_子图.py` + `09_子图_jxsd.py`、
> `14_子图持久化_官方补充.py`、`11_容错与测试_官方补充.py`。
> 前四者是课案（原版最短实现 + `_jxsd` 完整版），后两者是官方文档缺口补充。

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 6 个源文件全部 0 次模型调用，不读 `.env`、不连模型、不起服务 |
| 依赖 | `langgraph` / `langchain-core`（本项目 venv 已装） |
| 密钥 | 无 |
| 前置服务 | 无 |
| 预计耗时 | < 10 秒 |

> **分段说明**：第 1 节（时间旅行）用 `MemorySaver` 存内存快照；第 2 节（子图）纯字符串拼接；
> 第 3 节（子图持久化）用 `InMemorySaver` + `AIMessage` 发消息；第 4 节（容错测试）是重试 / 超时 / 断言。
> 四节都没有任何大模型调用，所以这一课能**离线秒跑、输出逐字节可复现**（只有 checkpoint_id 这类 uuid 每次不同）。
> 它和上一课 `02_记忆_短期与长期` 的关键区别：那里引入了「记忆 / 模型」，这里纯粹在玩图的调度与状态。

## 本节地图

四个主题按「从图内部到图之间」的顺序推进：

```mermaid
graph TB
    A["时间旅行<br/>checkpoint 快照回退"] --> B["子图<br/>图嵌图 + 键透传"]
    B --> C["子图持久化<br/>per-invocation / per-thread / stateless"]
    C --> D["容错与测试<br/>RetryPolicy / 超时 / 三粒度断言"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 主题 | 核心问题 | 答案（本节） |
|---|---|---|
| 时间旅行 | 跑错了能不能倒回去改？ | 能：`get_state_history` 看录像带，`update_state` 选帧回退，`invoke(None)` 重跑 |
| 子图 | 大图太乱怎么办？ | 拆成子图，先 `compile` 再当节点用；同名键自动透传，异名键互不可见 |
| 子图持久化 | 子代理要不要记得上一轮？ | 三档：默认 per-invocation（每次全新但单次内可恢复）/ per-thread（跨调用累积）/ stateless（最省） |
| 容错与测试 | 图跑不稳、跑不对怎么办？ | 节点级 `RetryPolicy` + 异步节点 `timeout`，再用三种粒度写断言 |

本课是 `01_langgraph/` 的收尾：`01` 建了第一张图，`02` 加了记忆，`03` 讲流式与中断，
这一课把「状态回溯 + 图组织 + 稳定性」三件工程化的事一次补全。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课同样用不到 `config`（全程离线、无模型），但这一格仍然保留 —— 一是保持全仓统一，
> 二是它顺便给出 `NB_DIR` / `WORKDIR` 两个变量。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

In [ ]:
# ===== 前置条件自检（本课离线：只需 langgraph / langchain-core，无需密钥与端口）=====
import importlib.metadata

for pkg in ("langgraph", "langchain-core"):
    try:
        print(f"  {pkg} {importlib.metadata.version(pkg)} ✔")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg} 缺失 —— 请先在本仓库 venv 里安装")

## 1. 时间旅行：回退到历史快照重跑

概念一句话：**每一步执行后自动保存检查点（checkpoint），通过 `get_state_history` 查看，
`update_state` 回退。**

怎么理解？把 checkpointer 想成一盘**录像带**：每执行一步就录一帧（一个 checkpoint），于是你随时可以：

1. `graph.get_state_history(config)` —— 把整盘带子倒出来看（**从新到旧**排列）；
2. 挑一帧，`graph.update_state(那一帧.config, 值)` —— 在那一帧上改写 / 打补丁；
3. `graph.invoke(None, 那一帧.config)` —— 从那一帧往后**重跑**。

用途：调试 Agent 行为（同一现场换个参数看它怎么走）、对比不同决策分支、线上出问题时回滚重来。
注意它是「**分叉**」而不是「抹掉」：重跑产生的新快照挂在被选快照后面，旧的那些依然在历史里。

### 1.1 课案原版：最短实现

课案原版只有 61 行，刚好把「查历史 → 挑一帧 → 回退重跑」三步各演示一次。
注意状态用的是裸 `StateGraph(dict)`（没有声明 reducer），所以 `log` 字段是**覆盖式**的 ——
后面 `08_时间旅行_jxsd` 用 `TypedDict` + 单字段 `count`，把这个细节再讲透。

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command


def step_1(state: dict) -> dict:
    return {"value": state["value"] + 1, "log": ["step_1"]}


def step_2(state: dict) -> dict:
    return {"value": state["value"] * 10, "log": ["step_2"]}


builder = StateGraph(dict)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", END)

graph = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "tt-1"}}

In [ ]:
# 正常执行一轮
result = graph.invoke({"value": 1, "log": []}, config)
print("正常执行结果：", result)

# ---------- 1. 查看全部历史快照 ----------
snapshots = list(graph.get_state_history(config))
for snap in snapshots:
    print(f"step={snap.metadata.get('step')}  值={snap.values}")

# ---------- 2. 选一个历史快照，从那里重新开始 ----------
# 选 step_1 之后的快照（value==2），修改输入后重新执行
# 注意：起始快照的 values 可能为 None，需要用 (s.values or {}) 防御
old_snapshot = [s for s in snapshots if (s.values or {}).get("value") == 2][0]
new_config = graph.update_state(
    old_snapshot.config,
    values={"value": 100},  # 改写历史状态
)

# 从这个快照继续执行（None 表示从当前位置继续跑后面的节点）
result = graph.invoke(None, new_config)
print("时间旅行后的结果：", result)

### 预期输出

```text
正常执行结果： {'value': 20, 'log': ['step_2']}
step=2  值={'value': 20, 'log': ['step_2']}
step=1  值={'value': 2, 'log': ['step_1']}
step=0  值={'value': 1, 'log': []}
step=-1  值=None
时间旅行后的结果： {'value': 1000, 'log': ['step_2']}
```

三个值得停一下的细节：

1. `step` 是**从大到小**的（2 → 1 → 0 → -1）—— 印证「history 从新到旧排列」；
2. `step=-1  值=None` 是**起始快照**：它只有 `config`（定位码），还没收到任何 state，
   所以 `values` 是 `None`，这也正是源码里要用 `(s.values or {})` 防御的原因；
3. `log` 只剩 `['step_2']` 一条 —— 因为裸 `StateGraph(dict)` 没有 reducer，字段是覆盖式的。

### 1.2 完整版：把「录像带」逐帧看清楚

课案原版只演示了「能回退」，完整版把课案两句**断言式结论**逐条打印验证：

- 「history 从新到旧排列」—— 实测 `metadata.step` 随列表下标单调递减；
- 「`next=()` 的表示已到 END，回退后 graph 认为已完成，不会再跑节点」。

关键字段：每个快照的 `source`（`input` = 刚收到 invoke 输入 / `loop` = 执行中的一帧 / `update` / `fork`）
和 `next`（`('add',)` = 卡在 add 之前，`()` = 已到 END）合起来，就是「这一帧处在什么时刻」的判据。

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

# 课案编号用的圈码字符（① ② ③ …），仅在打印时用于和课案原文对照
CIRCLED = "①②③④⑤⑥⑦⑧⑨⑩"


class State(TypedDict):
    count: int


def add(state: State) -> dict:
    return {"count": state["count"] + 1}


builder = StateGraph(State)
builder.add_node("add", add)
builder.add_edge(START, "add")
builder.add_edge("add", END)
graph = builder.compile(checkpointer=MemorySaver())  # 开启才有 checkpoint

config = {"configurable": {"thread_id": "1"}}

In [ ]:
def dump_history(title: str) -> list:
    """打印全部快照（列表顺序 = 从新到旧），并返回 history 列表。"""
    history = list(graph.get_state_history(config))
    n = len(history)
    print(title)
    print(f"  {'列表下标':<12}{'课案编号':<10}{'count':<8}{'next':<18}{'step':<6}{'source':<8}说明")
    print("  " + "-" * 92)
    for i, snap in enumerate(history):
        # 课案编号是「执行顺序」：最早的是 ①，最新的是 ⑥。
        # 而列表是「从新到旧」，所以两者正好反过来：课案编号 = n - i
        label = CIRCLED[n - 1 - i] if n - 1 - i < len(CIRCLED) else f"#{n - i}"
        count = snap.values.get("count") if snap.values else None
        nexts = str(snap.next)
        note = ""
        if nexts == "()":
            note = "已到 END，没有待执行节点"
        elif nexts == "('__start__',)":
            note = "刚收到 invoke 输入，还没进图"
        elif nexts == "('add',)":
            note = "卡在 add 节点之前（回退后能继续跑）"
        print(
            f"  history[{i}]".ljust(14)
            + f"{label}".ljust(12)
            + f"{str(count):<8}{nexts:<18}"
            + f"{str(snap.metadata.get('step')):<6}{str(snap.metadata.get('source')):<8}{note}"
        )
    return history

In [ ]:
print("=" * 100)
print("① 同一 thread_id 连续 invoke，checkpoint 链式累积")
print("=" * 100)

graph.invoke({"count": 0}, config)  # 快照①→②→③: count 0→1
result1 = graph.invoke({"count": 5}, config)  # 快照④→⑤→⑥: count 5→6
# 每次 invoke 产生 3 个快照（课案原话：「START 后、节点执行后、END 后」）
print("  第二次 invoke 的返回：", result1)  # 预期 {'count': 6}
print()

print("=" * 100)
print("② get_state_history(config)：把 6 个快照逐个列出来")
print("=" * 100)
history = dump_history("  快照清单：")
print()
print("  【实证 1】history 从新到旧排列：")
print(f"      history[0] 的 metadata.step = {history[0].metadata.get('step')}（最大，最新）")
print(f"      history[-1] 的 metadata.step = {history[-1].metadata.get('step')}（最小，最早）")
print("      step 单调递减 → 列表确实是从新到旧；课案编号①在最下面，就是这么来的。")
print()
print("  【实证 2】next=() 表示已到 END：")
ends = [f"history[{i}]" for i, s in enumerate(history) if not s.next]
print(f"      next 为空元组的快照：{ends}")
print(f"      它们对应的 count 值：{[history[i].values.get('count') for i, s in enumerate(history) if not s.next]}")
print("      两次 invoke 各有一个「跑完」的快照（count=1 和 count=6），它们的 next 都是 ()。")
print("      课案原话的后半句：这种快照回退后 graph 认为已完成，不会再跑节点。")
print()

### 预期输出

```text
① 同一 thread_id 连续 invoke，checkpoint 链式累积
  第二次 invoke 的返回： {'count': 6}

② get_state_history(config)：把 6 个快照逐个列出来
  快照清单：
  列表下标        课案编号      count   next              step  source  说明
  --------------------------------------------------------------------------------------------
  history[0]  ⑥           6       ()                4     loop    已到 END，没有待执行节点
  history[1]  ⑤           5       ('add',)          3     loop    卡在 add 节点之前（回退后能继续跑）
  history[2]  ④           1       ('__start__',)    2     input   刚收到 invoke 输入，还没进图
  history[3]  ③           1       ()                1     loop    已到 END，没有待执行节点
  history[4]  ②           0       ('add',)          0     loop    卡在 add 节点之前（回退后能继续跑）
  history[5]  ①           None    ('__start__',)    -1    input   刚收到 invoke 输入，还没进图

  【实证 1】history 从新到旧排列：
      history[0] 的 metadata.step = 4（最大，最新）
      history[-1] 的 metadata.step = -1（最小，最早）
      step 单调递减 → 列表确实是从新到旧；课案编号①在最下面，就是这么来的。

  【实证 2】next=() 表示已到 END：
      next 为空元组的快照：['history[0]', 'history[3]']
      它们对应的 count 值：[6, 1]
      两次 invoke 各有一个「跑完」的快照（count=1 和 count=6），它们的 next 都是 ()。
      课案原话的后半句：这种快照回退后 graph 认为已完成，不会再跑节点。
```

两次 `invoke` 各产生 3 个快照（START 后、节点执行后、END 后），共 6 个；
列表下标和课案编号（①…⑥）正好**反序**，这是「按执行顺序读」和「按列表读」最容易对不上的地方。

In [ ]:
print("=" * 100)
print("③ 选快照②回退：update_state + invoke(None, target.config)")
print("=" * 100)

# 挑一个 next=('add',) 的快照回退，graph 会从 add 继续执行
target = history[4]  # 快照②: count=0, next=('add',)
if target.next != ("add",):
    # 兜底：万一将来 LangGraph 的快照顺序变了，按内容找，绝不静默用错快照
    print(f"  ⚠️ history[4] 的 next={target.next}，与课案描述不符，按内容重新定位快照②")
    target = next(s for s in history if s.next == ("add",) and s.values.get("count") == 0)

print(f"  选中的快照：count={target.values.get('count')}, next={target.next}")
print(f"  它的 config：{target.config['configurable']['checkpoint_id']}（checkpoint_id 就是这一帧的定位码）")
print()

# update_state：在这一帧上「改写状态」。课案传的是 target.values（原值，等于不改内容，
# 只是把这一帧**重新激活**成一个新的分支点）；传别的值就是真的改写历史。
new_config = graph.update_state(target.config, target.values)  # 回退到 count=0
print(f"  update_state 完成，返回的新分支 config：{new_config['configurable']['checkpoint_id']}")
print(f"  此时 graph.get_state(new_config).next = {graph.get_state(new_config).next}")
print("  ↑ update_state 产出的是一张**新快照**（新 checkpoint_id），不是把旧快照改掉——")
print("    所以「时间旅行」是分叉，历史记录本身不会被破坏。")
print()

result = graph.invoke(None, target.config)  # 重新执行 add: 0 → 1
print("  invoke(None, target.config) →", result)
print(f"  result['count'] = {result['count']}")  # 1
print()

# ---------- 同一个回退点，换一套参数再跑一遍 ----------
result = graph.invoke({"count": 3}, target.config)
print("  invoke({'count': 3}, target.config) →", result)
print(f"  result['count'] = {result['count']}")  # 4
print("  ↑ 同一个历史快照，喂不同输入就走出不同分支——这就是「对比不同决策」的用法。")
print()

### 预期输出

```text
③ 选快照②回退：update_state + invoke(None, target.config)
  选中的快照：count=0, next=('add',)
  它的 config：<每次运行都不同的 uuid>（checkpoint_id 就是这一帧的定位码）

  update_state 完成，返回的新分支 config：<每次运行都不同的 uuid>
  此时 graph.get_state(new_config).next = ('add',)
  ↑ update_state 产出的是一张**新快照**（新 checkpoint_id），不是把旧快照改掉——
    所以「时间旅行」是分叉，历史记录本身不会被破坏。

  invoke(None, target.config) → {'count': 1}
  result['count'] = 1

  invoke({'count': 3}, target.config) → {'count': 4}
  result['count'] = 4
  ↑ 同一个历史快照，喂不同输入就走出不同分支——这就是「对比不同决策」的用法。
```

两处 `<...>` 是 checkpoint_id，**每次运行都不同**（uuid），不必在意 ——
它就是「这一帧录像带」的定位码。注意 `update_state` 返回的是**新** config，不是把旧快照改掉。

In [ ]:
print("=" * 100)
print("④ 回退/重跑之后，历史变成什么样？（回答「时间旅行会不会毁掉原来记录」）")
print("=" * 100)
after = dump_history("  当前快照清单：")
print()
print(f"  跑之前 {len(history)} 个快照，跑之后 {len(after)} 个 —— 只增不减：")
print("  旧快照全部保留，新产生的重跑快照挂在被选中快照的后面（形成分支）。")
print(f"  最新一帧 history[0]：count={after[0].values.get('count')}, next={after[0].next}")
print()
print("=" * 100)
print("小结：get_state_history 看录像带（新→旧），update_state 选中一帧打补丁，")
print("      invoke(None, 那一帧.config) 从那一帧往后重跑；历史只增不减，回退等于分叉。")
print("=" * 100)

### 预期输出

```text
④ 回退/重跑之后，历史变成什么样？（回答「时间旅行会不会毁掉原来记录」）
  当前快照清单：
  列表下标        课案编号      count   next              step  source  说明
  --------------------------------------------------------------------------------------------
  history[0]  #12         4       ()                3     loop    已到 END，没有待执行节点
  history[1]  #11         3       ('add',)          2     loop    卡在 add 节点之前（回退后能继续跑）
  history[2]  ⑩           0       ('__start__',)    1     input   刚收到 invoke 输入，还没进图
  history[3]  ⑨           1       ()                2     loop    已到 END，没有待执行节点
  history[4]  ⑧           0       ('add',)          1     fork    卡在 add 节点之前（回退后能继续跑）
  history[5]  ⑦           0       ('add',)          1     update  卡在 add 节点之前（回退后能继续跑）
  history[6]  ⑥           6       ()                4     loop    已到 END，没有待执行节点
  history[7]  ⑤           5       ('add',)          3     loop    卡在 add 节点之前（回退后能继续跑）
  history[8]  ④           1       ('__start__',)    2     input   刚收到 invoke 输入，还没进图
  history[9]  ③           1       ()                1     loop    已到 END，没有待执行节点
  history[10] ②           0       ('add',)          0     loop    卡在 add 节点之前（回退后能继续跑）
  history[11] ①           None    ('__start__',)    -1    input   刚收到 invoke 输入，还没进图

  跑之前 6 个快照，跑之后 12 个 —— 只增不减：
  旧快照全部保留，新产生的重跑快照挂在被选中快照的后面（形成分支）。
  最新一帧 history[0]：count=4, next=()
```

快照从 6 个涨到 12 个，**旧的一帧不少**：这就是「时间旅行 = 分叉，不是抹掉」的直接证据。
圈码只准备了 ①～⑩，超出的编号自动退化成 `#11` / `#12`（源码里的兜底分支）。

## 2. 子图：把编译好的图当节点用

课案原句：**将一个已编译好的图作为另一个图的节点，父图和子图各自独立编译。**

为什么要把图拆开？因为复杂 Agent 的图会长到没法看。拆成子图之后：

- 每个子图只负责一件事，可以**单独编译、单独运行、单独测试**；
- 父图只看「大流程」，不用被几十个内部节点淹没；
- 子图可以复用：同一份「检索子图」挂到多个父图上。

三个必须记住的点：子图先 `compile()` 再 `add_node`；**同名 state 键自动透传**；
子图只看得到**自己 schema 里声明过的键**（父图独有的键在子图内部「不存在」）。

### 2.1 课案原版：最短实现

课案原版用「规划器子图 → 执行器 → 审核」模拟一个三步流水线：子图负责产出 `plan`，
父图的 `executor` 读到 `plan` 再产出 `answer` —— 数据靠**同名 state 键**无缝传递。

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    question: str   # 用户问题
    plan: str       # 子图产出的计划
    answer: str     # 最终答案


# ---------- 子图：规划器 ----------
def planner(state: State) -> dict:
    """规划节点：假装调用 LLM 生成执行计划"""
    print(f"[规划器] 分析问题：{state['question']}")
    return {"plan": f"1. 检索资料 2. 归纳要点（针对：{state['question']}）"}


planner_builder = StateGraph(State)
planner_builder.add_node("planner", planner)
planner_builder.add_edge(START, "planner")
planner_builder.add_edge("planner", END)

# 子图先编译，再当节点使用
planner_graph = planner_builder.compile()


# ---------- 父图 ----------
def executor(state: State) -> dict:
    """执行节点：读取子图写入 plan，产出答案"""
    print(f"[执行器] 按计划执行：{state['plan']}")
    return {"answer": "（已按计划完成的答案）"}


def reviewer(state: State) -> dict:
    print(f"[审核] 检查答案：{state['answer']}")
    return {}


main_builder = StateGraph(State)
# 关键：子图作为一个普通节点加进父图
main_builder.add_node("planner", planner_graph)
main_builder.add_node("executor", executor)
main_builder.add_node("reviewer", reviewer)
main_builder.add_edge(START, "planner")
main_builder.add_edge("planner", "executor")
main_builder.add_edge("executor", "reviewer")
main_builder.add_edge("reviewer", END)

main_graph = main_builder.compile()

result = main_graph.invoke({"question": "LangGraph 是什么？"})
print("最终结果：", result)

### 预期输出

```text
[规划器] 分析问题：LangGraph 是什么？
[执行器] 按计划执行：1. 检索资料 2. 归纳要点（针对：LangGraph 是什么？）
[审核] 检查答案：（已按计划完成的答案）
最终结果： {'question': 'LangGraph 是什么？', 'plan': '1. 检索资料 2. 归纳要点（针对：LangGraph 是什么？）', 'answer': '（已按计划完成的答案）'}
```

关键一行是 `main_builder.add_node("planner", planner_graph)`：节点名是 `planner`，
但传进去的**不是函数，而是一张已编译的图**。这就是「子图当节点用」的全部秘密。

### 2.2 完整版：独立运行、异名键、subgraphs 命名空间

课案原版只给了最小示例，完整版补上「所以呢」—— 四件事：

1. 课案原文最小示例（同名键透传）；
2. 子图能**独立 compile → 独立 invoke**，这是「单独测试」的工程价值；
3. 父子 schema 不同名时的实测：**子图看不见父图独有的键**；
4. `stream(subgraphs=True)`：命名空间元组怎么标出「哪一层图的哪个节点」。

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    text: str


def prefix_node(state: State) -> dict:
    """子图里的节点：给文本加个前缀（原课案是 lambda，这里改成具名函数加打印观察点）。"""
    print(f"      [子图 prefix 节点] 收到的 state：{state}")
    return {"text": "[子图]" + state["text"]}


# 子图：加前缀
child = StateGraph(State)
child.add_node("prefix", prefix_node)
child.add_edge(START, "prefix")
child.add_edge("prefix", END)
child_graph = child.compile()  # 子图先独立编译


def suffix_node(state: State) -> dict:
    """父图自己的节点：注意它读到的是子图刚写进去的 text。"""
    print(f"      [父图 suffix 节点] 收到的 state：{state}")
    return {"text": state["text"] + " → 父图"}


# 父图：调用子图 + 自己的节点
parent = StateGraph(State)
parent.add_node("child", child_graph)  # 子图作为普通节点使用
parent.add_node("suffix", suffix_node)
parent.add_edge(START, "child")  # 先走子图
parent.add_edge("child", "suffix")  # 再走父图节点
parent.add_edge("suffix", END)
parent_graph = parent.compile()

In [ ]:
# 补充二：父图与子图共享同名 state 键 —— 以及「不共享」时会怎样
class ChildOnlyText(TypedDict):
    """子图 schema：只声明 text。"""

    text: str


class ParentWithNote(TypedDict):
    """父图 schema：比子图多一个 note。"""

    text: str
    note: str


def note_child_node(state: ChildOnlyText) -> dict:
    """故意在子图里打印「它到底能看见哪些键」。"""
    print(f"      [子图] 收到的键：{sorted(state.keys())}")
    return {"text": "[子图]" + state["text"]}


def note_after_node(state: ParentWithNote) -> dict:
    print(f"      [父图 after] 收到：{state}")
    return {"note": state["note"] + "（父图改过）"}


nc = StateGraph(ChildOnlyText)
nc.add_node("prefix", note_child_node)
nc.add_edge(START, "prefix")
nc.add_edge("prefix", END)
note_child_graph = nc.compile()

np_builder = StateGraph(ParentWithNote)
np_builder.add_node("child", note_child_graph)
np_builder.add_node("after", note_after_node)
np_builder.add_edge(START, "child")
np_builder.add_edge("child", "after")
np_builder.add_edge("after", END)
note_parent_graph = np_builder.compile()

In [ ]:
print("=" * 74)
print("① 课案原文：父图 invoke({'text': 'hello'})")
print("=" * 74)
print("  执行过程：")
result = parent_graph.invoke({"text": "hello"})
print(f"\n  完整返回：{result}")
print(f"  result['text'] = {result['text']}")  # 预期 [子图]hello → 父图
print("  ↑ 父图只声明了 text 一个字段，子图写进去的值直接就被父图节点读到了：")
print("    这就是「共享同名 state 键」的效果——不需要任何胶水代码。")
print()

print("=" * 74)
print("② 补充：子图能独立运行，所以能独立测试")
print("=" * 74)
child_result = child_graph.invoke({"text": "单独测试"})
print(f"  child_graph.invoke({{'text': '单独测试'}}) → {child_result}")
print("  ↑ 调试子图时不用把整张父图跑起来；同一个子图也能挂到别的父图上复用。")
print()

### 预期输出

```text
① 课案原文：父图 invoke({'text': 'hello'})
  执行过程：
      [子图 prefix 节点] 收到的 state：{'text': 'hello'}
      [父图 suffix 节点] 收到的 state：{'text': '[子图]hello'}

  完整返回：{'text': '[子图]hello → 父图'}
  result['text'] = [子图]hello → 父图
  ↑ 父图只声明了 text 一个字段，子图写进去的值直接就被父图节点读到了：
    这就是「共享同名 state 键」的效果——不需要任何胶水代码。

② 补充：子图能独立运行，所以能独立测试
      [子图 prefix 节点] 收到的 state：{'text': '单独测试'}
  child_graph.invoke({'text': '单独测试'}) → {'text': '[子图]单独测试'}
  ↑ 调试子图时不用把整张父图跑起来；同一个子图也能挂到别的父图上复用。
```

注意 ① 里子图 `prefix` 节点收到的 state 就是父图完整 state（`{'text': 'hello'}`）——
没有任何胶水代码，这就是「同名键自动透传」最直接的证据。

In [ ]:
print("=" * 74)
print("③ 补充：父子 schema 不同名时会怎样（实测）")
print("=" * 74)
print("  父图 schema：text + note；子图 schema：只有 text")
print("  执行过程：")
res = note_parent_graph.invoke({"text": "hello", "note": "初始 note"})
print(f"\n  完整返回：{res}")
print("  ↑ 观察两点：")
print("    1. 子图里打印出来的键**只有 ['text']** —— 父图的 note 在子图内部根本看不见，")
print("       因为子图只按自己的 schema 取键。别在子图里读写父图独有的字段。")
print("    2. 子图改的 text 照常合并回父图，父图独有的 note 原封不动地保留下来，")
print("       最后由父图自己的 after 节点修改。")
print()

### 预期输出

```text
③ 补充：父子 schema 不同名时会怎样（实测）
  父图 schema：text + note；子图 schema：只有 text
  执行过程：
      [子图] 收到的键：['text']
      [父图 after] 收到：{'text': '[子图]hello', 'note': '初始 note'}

  完整返回：{'text': '[子图]hello', 'note': '初始 note（父图改过）'}
  ↑ 观察两点：
    1. 子图里打印出来的键**只有 ['text']** —— 父图的 note 在子图内部根本看不见，
       因为子图只按自己的 schema 取键。别在子图里读写父图独有的字段。
    2. 子图改的 text 照常合并回父图，父图独有的 note 原封不动地保留下来，
       最后由父图自己的 after 节点修改。
```

两条结论一次跑全验证了：子图内部只有 `['text']`（父图的 `note` 在子图里「不存在」），
但子图改的 `text` 照常合并回父图、`note` 原封不动保留到最后。

In [ ]:
print("=" * 74)
print("④ 补充：stream(subgraphs=True) —— 连子图内部的执行都能看见")
print("=" * 74)
print("  每个事件的第一个元素是**命名空间元组**：() 表示父图，('child:xxx',) 表示进了子图。")
for namespace, chunk in parent_graph.stream({"text": "hello"}, subgraphs=True):
    where = "父图" if not namespace else f"子图 {namespace[0].split(':')[0]}"
    print(f"      [{where}] {chunk}")
print()
print("  ↑ 节点名相同也不会混：命名空间把「哪一层图的哪个节点」标得清清楚楚。")
print()
print("=" * 74)
print("小结：子图 = 先 compile、再当节点用；同名键自动透传，异名键互不可见；")
print("      拆分后每块都能独立跑、独立测，这是复杂 Agent 唯一可控的组织方式。")
print("=" * 74)

### 预期输出

```text
④ 补充：stream(subgraphs=True) —— 连子图内部的执行都能看见
  每个事件的第一个元素是**命名空间元组**：() 表示父图，('child:xxx',) 表示进了子图。
      [子图 prefix 节点] 收到的 state：{'text': 'hello'}
      [子图 child] {'prefix': {'text': '[子图]hello'}}
      [父图] {'child': {'text': '[子图]hello'}}
      [父图 suffix 节点] 收到的 state：{'text': '[子图]hello'}
      [父图] {'suffix': {'text': '[子图]hello → 父图'}}

  ↑ 节点名相同也不会混：命名空间把「哪一层图的哪个节点」标得清清楚楚。
```

`subgraphs=True` 让子图内部事件也流出来；命名空间元组的第一个元素形如 `('child:<uuid>',)`，
源码里用 `namespace[0].split(':')[0]` 把 uuid 剥掉只留 `child` —— 所以输出里**看不到** uuid。

## 3. 官方补充：子图持久化三档

来自 `14_子图持久化_官方补充.py`，对照官方文档 use-subgraphs 的「Subgraph persistence」一节。
它回答一个很实际的问题：**子代理要不要记住上一轮？** 客服机器人把问题转给「账单专家」子代理时，
这位专家该记得客户之前问过什么，还是每次都从零开始？

官方给的三个档位（`compile()` 的 `checkpointer` 参数）：

| 模式 | checkpointer= | 行为 |
|---|---|---|
| per-invocation | `None`（默认） | 每次调用**全新开始**；但在**单次调用内**继承父图 checkpointer，仍支持 interrupt 与持久执行 |
| per-thread | `True` | 状态**跨调用累积**（同一个 thread 上接着上次继续） |
| stateless | `False` | **完全没有检查点** —— 像普通函数调用，不支持 interrupt、无持久执行 |

前提：**父图必须带 checkpointer**，子图的持久化能力才谈得上（官方 Note）。

### 3.1 Demo 1：三档对照 —— 同一个 thread 连调三次看子图计数

关键设计：计数写在**子图私有字段** `run_count` 上（父图 schema 里没有这个键）。
如果父图也有同名字段并有 checkpointer，父图会把它一起存下来、再喂回子图，
于是「per-invocation 每次都是 1」这个差异就被掩盖了 —— 这是上一版实验设计错的地方。

In [ ]:
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt


class SubState(MessagesState):
    run_count: int          # 子图私有：本轮是子图的第几次执行


def sub_step(state: SubState) -> dict:
    count = state.get("run_count", 0) + 1
    return {"run_count": count, "messages": [AIMessage(content=f"子图第 {count} 次执行")]}


def build_subgraph(mode: str):
    """按官方三种模式编译子图。"""
    builder = StateGraph(SubState)
    builder.add_node("step", sub_step)
    builder.add_edge(START, "step")
    builder.add_edge("step", END)
    if mode == "per-thread":
        return builder.compile(checkpointer=True)      # 跨调用累积
    if mode == "stateless":
        return builder.compile(checkpointer=False)     # 完全无检查点
    return builder.compile()                           # per-invocation（默认）


class ParentState(MessagesState):
    """父图状态：只有 messages（**故意不加 run_count**，见上面的关键设计）。"""


def build_parent(mode: str):
    """父图：带 checkpointer，把子图当一个节点。"""
    builder = StateGraph(ParentState)
    builder.add_node("sub", build_subgraph(mode))
    builder.add_edge(START, "sub")
    builder.add_edge("sub", END)
    return builder.compile(checkpointer=InMemorySaver())


def demo_1_three_modes() -> None:
    print("=" * 70)
    print("Demo 1：三档对照（同一 thread 连调三次，看子图计数）")
    print("=" * 70)

    for mode in ("per-invocation", "per-thread", "stateless"):
        parent = build_parent(mode)
        config = {"configurable": {"thread_id": f"sub-persist-{mode}"}}
        counts: list[str] = []
        for _ in range(3):
            result = parent.invoke({"messages": []}, config)
            # 三次调用的消息会累积在父图里；取出「子图第 N 次执行」这条
            latest = [m.content for m in result["messages"] if "子图第" in str(m.content)]
            counts.append(str(latest[-1]).split("第")[1].split("次")[0].strip() if latest else "?")
        print(f"  {mode:<15} 三次调用中子图报告的批次：{counts}")
        if mode == "per-thread":
            print("                  ↑ 1→2→3：**状态跨调用累积**（每次接着上次继续）")
        else:
            print("                  ↑ 每次都从 1 开始：说明子图状态没有跨调用保留")

    print(
        "\n  ↑ 结论清清楚楚：\n"
        "    · per-invocation（默认）：每次调用**全新开始**（计数恒为 1）；\n"
        "    · per-thread（checkpointer=True）：**跨调用累积**（1→2→3）；\n"
        "    · stateless（checkpointer=False）：也每次从 1 开始 —— 它与 per-invocation\n"
        "      **在这一栏看不出区别**，差别在 Demo 2（能不能中断恢复）。"
    )


demo_1_three_modes()

### 预期输出

```text
Demo 1：三档对照（同一 thread 连调三次，看子图计数）
  per-invocation  三次调用中子图报告的批次：['1', '1', '1']
                  ↑ 每次都从 1 开始：说明子图状态没有跨调用保留
  per-thread      三次调用中子图报告的批次：['1', '2', '3']
                  ↑ 1→2→3：**状态跨调用累积**（每次接着上次继续）
  stateless       三次调用中子图报告的批次：['1', '1', '1']
                  ↑ 每次都从 1 开始：说明子图状态没有跨调用保留

  ↑ 结论清清楚楚：
    · per-invocation（默认）：每次调用**全新开始**（计数恒为 1）；
    · per-thread（checkpointer=True）：**跨调用累积**（1→2→3）；
    · stateless（checkpointer=False）：也每次从 1 开始 —— 它与 per-invocation
      **在这一栏看不出区别**，差别在 Demo 2（能不能中断恢复）。
```

三档里只有 **per-thread** 是 `1→2→3`（跨调用累积），其余两档每次都从 1 开始。
per-invocation 和 stateless **在这一栏看不出区别** —— 真正的差别在 Demo 2 / Demo 3。

### 3.2 Demo 2：per-invocation vs stateless 的真正差异 —— 能否 interrupt

官方对 per-invocation 的描述是「在**单次调用内**继承父图 checkpointer，因此仍支持
interrupt 与持久执行」；而 stateless 是「完全没有检查点」。本 Demo 让子图里的节点调用
`interrupt()`，对两种模式各跑一次看结果。

In [ ]:
class InterruptSubState(MessagesState):
    approved: str


def sub_step_with_interrupt(state: InterruptSubState) -> dict:
    decision = interrupt({"question": "子图需要人工确认，批准吗？"})
    return {"approved": str(decision), "messages": [AIMessage(content=f"子图收到决定：{decision}")]}


def build_interrupt_subgraph(mode: str):
    builder = StateGraph(InterruptSubState)
    builder.add_node("ask", sub_step_with_interrupt)
    builder.add_edge(START, "ask")
    builder.add_edge("ask", END)
    if mode == "stateless":
        return builder.compile(checkpointer=False)
    return builder.compile()          # per-invocation：继承父图 checkpointer


def build_interrupt_parent(mode: str):
    builder = StateGraph(ParentState)
    builder.add_node("sub", build_interrupt_subgraph(mode))
    builder.add_edge(START, "sub")
    builder.add_edge("sub", END)
    return builder.compile(checkpointer=InMemorySaver())


def demo_2_interrupt_difference() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：per-invocation vs stateless —— 能不能在子图里中断")
    print("=" * 70)

    observed: dict[str, str] = {}
    for mode in ("per-invocation", "stateless"):
        print(f"\n  --- 模式：{mode} ---")
        parent = build_interrupt_parent(mode)
        config = {"configurable": {"thread_id": f"sub-interrupt-{mode}"}}
        try:
            # 注意：approved 是**子图私有字段**，父图 schema 里没有 → 传了也会被静默丢弃，
            # 这里传它只是为了让调用形状与子图对齐（真正生效的是 interrupt 的返回值）。
            first = parent.invoke({"messages": []}, config)
            if "__interrupt__" in first:
                payload = first["__interrupt__"][0].value
                print(f"    第一次 invoke 正常暂停，中断负载：{payload}")
                resumed = parent.invoke(Command(resume="批准"), config)
                print(f"    恢复后的消息：{[str(m.content) for m in resumed['messages']]}")
                observed[mode] = "可中断可恢复"
            else:
                print(f"    没有中断，直接跑完：{[str(m.content) for m in first['messages']]}")
                observed[mode] = "未中断"
        except Exception as exc:  # noqa: BLE001
            print(f"    ✘ 执行失败：{type(exc).__name__}: {str(exc)[:160]}")
            observed[mode] = f"失败：{type(exc).__name__}"

    print(f"\n  实测结果对照：{observed}")
    if observed.get("stateless") == "可中断可恢复":
        print(
            "  ⚠️ 注意：**本机这个实验里 stateless 也能中断恢复** —— 与官方那句\n"
            "     「stateless 不支持 interrupts / durable execution」并不冲突，而是场景问题：\n"
            "     这里**父图带了 checkpointer**，中断信息由父图落盘，所以照样能暂停恢复。\n"
            "     stateless 的代价要在别的场景才显形：子图**内部多步执行**、中途崩了要续跑时\n"
            "     它没有自己的检查点可用（本文件没有构造那个场景，所以不下断言）。\n"
            "     实践建议：**没特殊理由就用默认的 per-invocation** —— 它等于\n"
            "     「每次全新 + 单次调用内可恢复」，两头都占了。"
        )
    print(
        "\n  ↑ 三档的完整画像（结合 Demo 1 的计数结果）：\n"
        "    · per-invocation：每次全新（计数恒 1）+ 单次调用内可中断/可恢复 ← 默认首选；\n"
        "    · per-thread：跨调用累积（1→2→3），适合需要多轮记忆的子代理；\n"
        "    · stateless：也每次全新；省掉子图自己的检查点，但**没有可下钻的子图状态**\n"
        "      （Demo 3 会看到：它的子图快照是拿不到的/没有的）。"
    )


demo_2_interrupt_difference()

### 预期输出

```text
Demo 2：per-invocation vs stateless —— 能不能在子图里中断

  --- 模式：per-invocation ---
    第一次 invoke 正常暂停，中断负载：{'question': '子图需要人工确认，批准吗？'}
    恢复后的消息：['子图收到决定：批准']

  --- 模式：stateless ---
    第一次 invoke 正常暂停，中断负载：{'question': '子图需要人工确认，批准吗？'}
    恢复后的消息：['子图收到决定：批准']

  实测结果对照：{'per-invocation': '可中断可恢复', 'stateless': '可中断可恢复'}
  ⚠️ 注意：**本机这个实验里 stateless 也能中断恢复** —— 与官方那句
     「stateless 不支持 interrupts / durable execution」并不冲突，而是场景问题：
     这里**父图带了 checkpointer**，中断信息由父图落盘，所以照样能暂停恢复。
     stateless 的代价要在别的场景才显形：子图**内部多步执行**、中途崩了要续跑时
     它没有自己的检查点可用（本文件没有构造那个场景，所以不下断言）。
     实践建议：**没特殊理由就用默认的 per-invocation** —— 它等于
     「每次全新 + 单次调用内可恢复」，两头都占了。

  ↑ 三档的完整画像（结合 Demo 1 的计数结果）：
    · per-invocation：每次全新（计数恒 1）+ 单次调用内可中断/可恢复 ← 默认首选；
    · per-thread：跨调用累积（1→2→3），适合需要多轮记忆的子代理；
    · stateless：也每次全新；省掉子图自己的检查点，但**没有可下钻的子图状态**
      （Demo 3 会看到：它的子图快照是拿不到的/没有的）。
```

实测：**两种模式都能中断并恢复** —— 因为父图带了 checkpointer，中断信息由父图落盘。
官方说 stateless「不支持 interrupts」，在本机这个场景下**没复现出差异**，代价要在 Demo 3 才显形。

### 3.3 Demo 3：下钻子图状态（运维/排障用）

实测：`get_state(config, subgraphs=True)` 返回 **StateSnapshot**，下钻路径是
`snapshot.tasks[*].state`（Task 上的 state 才是子图快照），**不是** `snapshot.subgraphs`
（不存在该属性）。注意**要在暂停时下钻**：图跑完后 tasks 是空的，看不到子图快照。

In [ ]:
def demo_3_inspect_subgraph_state() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：下钻子图状态（在中断暂停时看最合适）")
    print("=" * 70)

    for mode in ("per-invocation", "stateless"):
        print(f"\n  --- 模式：{mode} ---")
        parent = build_interrupt_parent(mode)
        config = {"configurable": {"thread_id": f"sub-inspect-{mode}"}}
        parent.invoke({"messages": []}, config)                     # 跑到中断处停下（父图无 approved 字段）

        snapshot = parent.get_state(config, subgraphs=True)
        print(f"    父图 values（只有 messages，子图私有字段不冒泡）：{sorted(snapshot.values.keys())}")
        print(f"    待执行任务数：{len(snapshot.tasks)}")
        for task in snapshot.tasks:
            inner = getattr(task, "state", None)
            print(f"      任务 {task.name}：", end="")
            if inner is None:
                print("（拿不到子图快照）")
                continue
            print(f"子图快照 ✔")
            print(f"        子图 values={inner.values}")
            print(f"        子图待执行节点={inner.next}")
            print(f"        子图内的中断={[item.value for item in (inner.interrupts or [])]}")
        parent.invoke(Command(resume="批准"), config)               # 收尾，别留半截状态

    print(
        "\n  ↑ 三个实测要点（都是这次踩出来的）：\n"
        "    ① 下钻路径是 **`snapshot.tasks[*].state`**，不是 `snapshot.subgraphs`（不存在该属性）；\n"
        "    ② **要在暂停时下钻** —— 图跑完后 tasks 为空，什么都看不到；\n"
        "    ③ 子图私有字段（比如本文件的 run_count）**不会冒泡到父图**，\n"
        "       父图 values 里只有 messages —— 想给上层用，得自己在子图里 return 出去。"
    )


demo_3_inspect_subgraph_state()
print("\n全部 Demo 执行完毕（0 次模型调用，离线可复现）。")

### 预期输出

```text
Demo 3：下钻子图状态（在中断暂停时看最合适）

  --- 模式：per-invocation ---
    父图 values（只有 messages，子图私有字段不冒泡）：['messages']
    待执行任务数：1
      任务 sub：子图快照 ✔
        子图 values={'messages': []}
        子图待执行节点=('ask',)
        子图内的中断=[{'question': '子图需要人工确认，批准吗？'}]

  --- 模式：stateless ---
    父图 values（只有 messages，子图私有字段不冒泡）：['messages']
    待执行任务数：1
      任务 sub：（拿不到子图快照）

  ↑ 三个实测要点（都是这次踩出来的）：
    ① 下钻路径是 **`snapshot.tasks[*].state`**，不是 `snapshot.subgraphs`（不存在该属性）；
    ② **要在暂停时下钻** —— 图跑完后 tasks 为空，什么都看不到；
    ③ 子图私有字段（比如本文件的 run_count）**不会冒泡到父图**，
       父图 values 里只有 messages —— 想给上层用，得自己在子图里 return 出去。

全部 Demo 执行完毕（0 次模型调用，离线可复现）。
```

**这才是 stateless 的实测差异**：per-invocation 暂停时能下钻到子图快照（values / next / interrupts），
stateless 拿不到（`task.state is None`，打印「拿不到子图快照」）。

## 4. 官方补充：容错与测试

来自 `11_容错与测试_官方补充.py`。课案把「怎么把图跑起来」讲透了，但没讲
「跑不稳怎么办」和「怎么证明它是对的」—— 容错和测试就是从 demo 到可用系统的分界线。

官方给出的**错误处理四分类**：

| # | 错误类型 | 谁来修 | 策略 | 本仓库对应 |
|---|---|---|---|---|
| 1 | 瞬时错误（网络/限流） | 系统 | 节点级 `RetryPolicy`（本节 Demo 1/2） | 本节 |
| 2 | LLM 可修复（工具失败） | 模型 | 错误转 ToolMessage 回灌模型 | `02_langchain` 官方补充 |
| 3 | 用户可修复（缺信息） | 人工 | `interrupt()` 问人 | `01_langgraph` 06/07 |
| 4 | 意外错误 | 开发者 | 让它抛出来，别吞 | —— |

本地版本的三条硬约束（**都是实测出来的，官方文档没明说**）：

- **A**：`timeout=` 只支持**异步节点**，同步节点上写直接抛 `ValueError`，必须 `async def` + `asyncio.run(ainvoke(...))`；
- **B**：想超时被重试，`retry_on` 要写 `NodeTimeoutError`（它**不是** `TimeoutError` 的子类），`except` 也要写 `except NodeTimeoutError`；
- **C**：默认 `RetryPolicy` **不重试** `ValueError` 这类业务异常（特性不是 bug）。

### 4.1 Demo 1：RetryPolicy —— 谁会重试，谁不会

节点级重试挂在 `add_node(..., retry_policy=RetryPolicy(...))` 上。字段（实测签名）：
`max_attempts`（含首次）、`initial_interval`、`backoff_factor`、`max_interval`、
`jitter`、`retry_on`（异常类/元组，或「接收异常返回 bool」的函数）。

In [ ]:
import asyncio
import time

import nest_asyncio

from typing_extensions import TypedDict

from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeTimeoutError
from langgraph.graph import END, START, StateGraph
from langgraph.types import RetryPolicy, TimeoutPolicy

# notebook 特有：ipykernel 内核已有一个运行中的事件循环，裸 asyncio.run 会抛
# 「cannot be called from a running event loop」；apply 后 asyncio.run 才能用（Demo 2 需要）。
nest_asyncio.apply()


class SimpleState(TypedDict):
    log: list


# ---- Part A：ValueError —— 默认策略**不**重试 ----
value_error_attempts = {"n": 0}


def raise_value_error(state: SimpleState) -> dict:
    """模拟「参数写错了」这类业务异常：重试没有意义。"""
    value_error_attempts["n"] += 1
    raise ValueError(f"第 {value_error_attempts['n']} 次：参数不合法")


graph_value_error = (
    StateGraph(SimpleState)
    .add_node("bad", raise_value_error, retry_policy=RetryPolicy(max_attempts=3))
    .add_edge(START, "bad")
    .add_edge("bad", END)
    .compile()
)


# ---- Part B：自定义瞬时异常 —— 会按策略重试并最终成功 ----
class TransientError(Exception):
    """模拟「网络抖了一下」：这种才值得重试。"""


transient_attempts = {"n": 0}


def flaky_call(state: SimpleState) -> dict:
    transient_attempts["n"] += 1
    if transient_attempts["n"] < 3:
        raise TransientError(f"第 {transient_attempts['n']} 次：下游服务暂时不可用")
    return {"log": [f"第 {transient_attempts['n']} 次调用成功"]}


graph_retry = (
    StateGraph(SimpleState)
    .add_node(
        "call_api",
        flaky_call,
        # 只重试我们指定的异常；initial_interval 给小值，教学时不必等
        retry_policy=RetryPolicy(
            max_attempts=3,
            retry_on=(TransientError,),
            initial_interval=0.05,
            backoff_factor=2.0,
            jitter=False,
        ),
    )
    .add_edge(START, "call_api")
    .add_edge("call_api", END)
    .compile()
)

In [ ]:
print("=" * 70)
print("Demo 1：RetryPolicy —— 谁会重试，谁不会")
print("=" * 70)

print("\nPart A：ValueError（业务异常）")
try:
    graph_value_error.invoke({"log": []})
    print("  没抛异常？（不符合预期）")
except ValueError as exc:
    print(f"  抛出 ValueError：{exc}")
    print(
        f"  实际尝试了 {value_error_attempts['n']} 次 ← 虽然配了 max_attempts=3，"
        "但默认策略**不重试** ValueError 这类业务异常（重试也没用，早失败早修）"
    )

print("\nPart B：TransientError（瞬时异常）")
result = graph_retry.invoke({"log": []})
print(f"  最终状态：{result['log']}")
print(
    f"  实际尝试了 {transient_attempts['n']} 次 ← 前两次抛异常被策略吸收，"
    "第三次成功；retry_on 显式指定了要重试的异常类型"
)

### 预期输出

```text
Demo 1：RetryPolicy —— 谁会重试，谁不会

Part A：ValueError（业务异常）
  抛出 ValueError：第 1 次：参数不合法
  实际尝试了 1 次 ← 虽然配了 max_attempts=3，但默认策略**不重试** ValueError 这类业务异常（重试也没用，早失败早修）

Part B：TransientError（瞬时异常）
  最终状态：['第 3 次调用成功']
  实际尝试了 3 次 ← 前两次抛异常被策略吸收，第三次成功；retry_on 显式指定了要重试的异常类型
```

`ValueError` 只尝试 1 次（默认策略**不重试**业务异常 —— 特性不是 bug）；
`TransientError` 显式写进 `retry_on`，尝试 3 次后成功。

### 4.2 Demo 2：节点超时 —— 只支持异步节点（实测硬约束）

为什么必须异步：同步 Python 代码没法在中途被安全打断，所以 langgraph 只在**异步执行路径**上
实现节点级超时。同步节点写 `timeout=` 会在 compile 期直接抛
`ValueError: Node timeouts are only supported for async nodes ...`。

In [ ]:
async def slow_async_node(state: SimpleState) -> dict:
    """模拟一个卡住的异步调用（真实场景：模型/接口迟迟不返回）。"""
    await asyncio.sleep(1.5)
    return {"log": ["慢节点居然跑完了（说明没被超时拦住）"]}


# ---- Part A：裸数字超时 ----
graph_timeout = (
    StateGraph(SimpleState)
    .add_node("slow", slow_async_node, timeout=0.4)   # 单位：秒（也接受 timedelta / TimeoutPolicy）
    .add_edge(START, "slow")
    .add_edge("slow", END)
    .compile()
)


# ---- Part B：TimeoutPolicy 精细控制（run_timeout 与 idle_timeout 谁先到算谁）----
graph_timeout_policy = (
    StateGraph(SimpleState)
    .add_node(
        "slow",
        slow_async_node,
        timeout=TimeoutPolicy(run_timeout=1.0, idle_timeout=0.25),
        # run_timeout ：单次尝试的总时长上限
        # idle_timeout：连续多久没有产出就算卡死（本例节点一直在 sleep，所以它先触发）
    )
    .add_edge(START, "slow")
    .add_edge("slow", END)
    .compile()
)


# ---- Part C：超时 + 重试的正确/错误写法对照 ----
timeout_retry_attempts = {"n": 0}


async def slow_always(state: SimpleState) -> dict:
    timeout_retry_attempts["n"] += 1
    await asyncio.sleep(1.0)      # 永远比 timeout 长 → 每次尝试都超时
    return {"log": ["不可能走到这里"]}


def build_timeout_retry_graph(retry_policy: RetryPolicy):
    return (
        StateGraph(SimpleState)
        .add_node("slow", slow_always, timeout=0.2, retry_policy=retry_policy)
        .add_edge(START, "slow")
        .add_edge("slow", END)
        .compile()
    )

In [ ]:
print("\n" + "=" * 70)
print("Demo 2：节点超时 —— 只支持异步节点（本地实测）")
print("=" * 70)

print("\nPart A：timeout=0.4（节点实际要 1.5 秒）")
started = time.time()
try:
    # 必须走异步入口：这条 async 节点若走同步 invoke，会直接报
    # TypeError: No synchronous function provided to "slow"；
    # （「同步节点 + timeout」是另一回事：那种写法在 compile 期就抛
    #  ValueError: Node timeouts are only supported for async nodes）
    asyncio.run(graph_timeout.ainvoke({"log": []}))
    print("  居然没超时？（不符合预期）")
except NodeTimeoutError as exc:
    print(f"  NodeTimeoutError（{time.time() - started:.2f}s）：{str(exc)[:70]}")
print("  注意上面用的是异步入口 asyncio.run(graph.ainvoke(...))；")
print("  同步节点写 timeout 会直接报 ValueError: Node timeouts are only supported for async nodes")

print("\nPart B：TimeoutPolicy(run_timeout=1.0, idle_timeout=0.25)")
started = time.time()
try:
    asyncio.run(graph_timeout_policy.ainvoke({"log": []}))
    print("  居然没超时？（不符合预期）")
except NodeTimeoutError:
    print(
        f"  NodeTimeoutError（{time.time() - started:.2f}s）← 约 0.25s 就炸了："
        "idle_timeout 先于 run_timeout 触发（节点一直没产出）"
    )

print("\nPart C：超时能不能被重试？三种 retry_on 写法的实测对照")
for label, policy in (
    ("retry_on=(TimeoutError,)     ", RetryPolicy(max_attempts=3, retry_on=(TimeoutError,), initial_interval=0.05)),
    ("retry_on=(NodeTimeoutError,) ", RetryPolicy(max_attempts=3, retry_on=(NodeTimeoutError,), initial_interval=0.05)),
    ("默认 RetryPolicy()           ", RetryPolicy(max_attempts=3, initial_interval=0.05)),
):
    timeout_retry_attempts["n"] = 0
    try:
        asyncio.run(build_timeout_retry_graph(policy).ainvoke({"log": []}))
        outcome = "成功"
    except NodeTimeoutError:
        outcome = "NodeTimeoutError"
    except Exception as exc:  # noqa: BLE001
        outcome = f"{type(exc).__name__}"
    print(f"  {label} → {outcome}，共尝试 {timeout_retry_attempts['n']} 次")
print(
    "  ↑ 关键坑：**`NodeTimeoutError` 不是 `TimeoutError` 的子类**\n"
    "    （实测 MRO：NodeTimeoutError → Exception → BaseException → object）；\n"
    "    所以写 `retry_on=(TimeoutError,)` 匹配不上、一次都不重试，\n"
    "    要重试超时就得写 `(NodeTimeoutError,)` 或者直接用默认 `RetryPolicy()`。\n"
    "    同理：**捕获也不能写 `except TimeoutError`**，要写 `except NodeTimeoutError` ——\n"
    "    否则超时会直接冒泡出去（这正是本 Demo 用 except NodeTimeoutError 的原因）。"
)

### 预期输出

```text
Demo 2：节点超时 —— 只支持异步节点（本地实测）

Part A：timeout=0.4（节点实际要 1.5 秒）
  NodeTimeoutError（0.41s）：Node 'slow' exceeded its run timeout of 0.400s (elapsed: 0.406s).
  注意上面用的是异步入口 asyncio.run(graph.ainvoke(...))；
  同步节点写 timeout 会直接报 ValueError: Node timeouts are only supported for async nodes

Part B：TimeoutPolicy(run_timeout=1.0, idle_timeout=0.25)
  NodeTimeoutError（0.26s）← 约 0.25s 就炸了：idle_timeout 先于 run_timeout 触发（节点一直没产出）

Part C：超时能不能被重试？三种 retry_on 写法的实测对照
  retry_on=(TimeoutError,)      → NodeTimeoutError，共尝试 1 次
  retry_on=(NodeTimeoutError,)  → NodeTimeoutError，共尝试 3 次
  默认 RetryPolicy()            → NodeTimeoutError，共尝试 3 次
  ↑ 关键坑：**`NodeTimeoutError` 不是 `TimeoutError` 的子类**
    （实测 MRO：NodeTimeoutError → Exception → BaseException → object）；
    所以写 `retry_on=(TimeoutError,)` 匹配不上、一次都不重试，
    要重试超时就得写 `(NodeTimeoutError,)` 或者直接用默认 `RetryPolicy()`。
    同理：**捕获也不能写 `except TimeoutError`**，要写 `except NodeTimeoutError` ——
    否则超时会直接冒泡出去（这正是本 Demo 用 except NodeTimeoutError 的原因）。
```

最关键的坑：`retry_on=(TimeoutError,)` **只尝试 1 次**（匹配不上），
换成 `(NodeTimeoutError,)` 或默认策略则尝试 3 次。原因是 `NodeTimeoutError` 不是 `TimeoutError` 的子类。

> 上面 `0.41s` / `0.406s` / `0.26s` 是实测耗时，**每次运行略有不同**（受计时器影响），
> 别逐字比对；报错文案 `Node 'slow' exceeded ...` 与重试次数（1 / 3 / 3）才是稳定的结论。

### 4.3 Demo 3：测试三模式（官方 test.mdx）

官方推荐的三种测试粒度（本文件用 `assert` 直接演示，放进 pytest 就是现成的 `test_*.py`）：

1. **整图测试**：`invoke` 一遍，断言最终 state；
2. **单节点测试**：`graph.nodes["名字"].invoke(...)`，绕过图和 checkpointer；
3. **部分执行**：`compile(interrupt_after=[...])` + `update_state(as_node=...)`，只测图中间某一段。

In [ ]:
class CalcState(TypedDict):
    value: int


def double(state: CalcState) -> dict:
    return {"value": state["value"] * 2}


graph_calc = (
    StateGraph(CalcState)
    .add_node("double", double)
    .add_edge(START, "double")
    .add_edge("double", END)
    .compile()
)


class TwoStepState(TypedDict):
    x: int
    y: int


def step_a(state: TwoStepState) -> dict:
    return {"x": state.get("x", 0) + 1}


def step_b(state: TwoStepState) -> dict:
    return {"y": state.get("x", 0) * 10}


# interrupt_after 是**静态断点**：图跑到 step_a 之后自动停下（课案 06 章讲过静态断点）
graph_two_step = (
    StateGraph(TwoStepState)
    .add_node("step_a", step_a)
    .add_node("step_b", step_b)
    .add_edge(START, "step_a")
    .add_edge("step_a", "step_b")
    .add_edge("step_b", END)
    .compile(checkpointer=MemorySaver(), interrupt_after=["step_a"])
)

In [ ]:
print("\n" + "=" * 70)
print("Demo 3：测试三模式（官方 test.mdx）")
print("=" * 70)

print("\n模式 1：整图测试 —— invoke 后断言最终 state")
whole = graph_calc.invoke({"value": 21})
assert whole == {"value": 42}, whole
print(f"  graph_calc.invoke({{'value': 21}}) == {{'value': 42}}  ✔ 断言通过（{whole}）")

print("\n模式 2：单节点测试 —— graph.nodes['名字'].invoke(...)，绕过图与 checkpointer")
node_output = graph_calc.nodes["double"].invoke({"value": 21})
assert node_output == {"value": 42}, node_output
print(f"  graph_calc.nodes['double'].invoke({{'value': 21}}) == {{'value': 42}}  ✔（{node_output}）")
print("  ↑ 节点函数是普通函数，能脱离图单独测 —— 复杂图排错时先这样定位到具体节点")

print("\n模式 3：部分执行 —— 静态断点 + update_state(as_node=...)，只测中间一段")
config = {"configurable": {"thread_id": "test-two-step"}}
first = graph_two_step.invoke({"x": 0, "y": 0}, config)
print(f"  interrupt_after=['step_a'] 第一次 invoke 停在 step_a 之后：{first}")
# as_node="step_a" 表示「假装这些状态是 step_a 刚产出的」，图会从 step_a 的下游继续
graph_two_step.update_state(config, {"x": 100}, as_node="step_a")
second = graph_two_step.invoke(None, config)
assert second == {"x": 100, "y": 1000}, second
print(f"  注入 x=100 后继续跑：{second}  ✔ step_b 按注入值算出 y=1000")
print(
    "  ↑ 本次是「先跑到 step_a 之后停下（step_a 已执行一次），再注入状态从 step_b 继续」，\n"
    "    所以断言的是**中间那一段**；官方 test.mdx 的更省版本是：先 update_state(as_node=\"step_a\")\n"
    "    伪造上游输出，再 invoke(None, config, interrupt_after=\"step_b\")，**全程不跑上游** ——\n"
    "    上游很慢/很贵时（比如要调模型）能把成本压到最低。"
)

print("\n全部 Demo 执行完毕（0 次模型调用，离线可复现）。")

### 预期输出

```text
Demo 3：测试三模式（官方 test.mdx）

模式 1：整图测试 —— invoke 后断言最终 state
  graph_calc.invoke({'value': 21}) == {'value': 42}  ✔ 断言通过（{'value': 42}）

模式 2：单节点测试 —— graph.nodes['名字'].invoke(...)，绕过图与 checkpointer
  graph_calc.nodes['double'].invoke({'value': 21}) == {'value': 42}  ✔（{'value': 42}）
  ↑ 节点函数是普通函数，能脱离图单独测 —— 复杂图排错时先这样定位到具体节点

模式 3：部分执行 —— 静态断点 + update_state(as_node=...)，只测中间一段
  interrupt_after=['step_a'] 第一次 invoke 停在 step_a 之后：{'x': 1, 'y': 0}
  注入 x=100 后继续跑：{'x': 100, 'y': 1000}  ✔ step_b 按注入值算出 y=1000
  ↑ 本次是「先跑到 step_a 之后停下（step_a 已执行一次），再注入状态从 step_b 继续」，
    所以断言的是**中间那一段**；官方 test.mdx 的更省版本是：先 update_state(as_node="step_a")
    伪造上游输出，再 invoke(None, config, interrupt_after="step_b")，**全程不跑上游** ——
    上游很慢/很贵时（比如要调模型）能把成本压到最低。

全部 Demo 执行完毕（0 次模型调用，离线可复现）。
```

`graph.nodes['double']` 是 dict，`.invoke()` 直接返回节点更新字典；
三个断言全部通过（`interrupt_after` + `update_state(as_node=...)` + `invoke(None, config)`）。

## 小结

- **时间旅行**靠 checkpointer：`get_state_history`（新→旧）看录像带，`update_state` 选帧回退，
  `invoke(None, 那一帧.config)` 重跑；历史只增不减，回退等于**分叉**；
- **子图** = 先 `compile`、再 `add_node`；同名 state 键自动透传，异名键互不可见；
  子图能独立跑独立测，`stream(subgraphs=True)` 用命名空间区分层级；
- **子图持久化**三档：默认 per-invocation（每次全新 + 单次内可恢复）最常用；
  per-thread 跨调用累积；stateless 省检查点但拿不到子图快照；
- **容错**：`RetryPolicy` 只重试你指定的异常（默认不碰 ValueError），
  节点 `timeout=` 只对异步节点生效；
- **测试**三种粒度：整图 / 单节点 / 部分执行（静态断点 + `update_state(as_node=...)`）。

## 常见坑

1. **时间旅行必须配 checkpointer**：没有 checkpointer 就没有 history，`get_state_history` 返回空列表；
2. **`get_state_history` 返回生成器**：用 `list()` 一次取出；对生成器做两次遍历会得到空结果；
3. **`update_state(config, values)` 的 values 是「这一帧的新值」不是补丁**：传原值等于只重新激活帧，传别的值才是真改写历史；
4. **`update_state` 返回新 config**：后续 `invoke` 要用它或原 `target.config`，混用会「明明回退了跑出来还是旧结果」；
5. **子图必须先 `compile` 再 `add_node`**：传未编译的 StateGraph 直接报错；
6. **子图看不见父图独有的键**：想在子图里读写父图字段，要么加进子图 schema，要么显式传参；
7. **`graph.get_graph(xray=True)` 才能展开子图内部**：默认 `get_graph()` 里子图只是一个方框；
8. **`NodeTimeoutError` 不是 `TimeoutError` 的子类**：`retry_on=(TimeoutError,)` 匹配不上、不重试，
   且不能用 `except TimeoutError` 捕获 —— 都要写 `NodeTimeoutError`；
9. **节点 `timeout=` 只支持异步节点**：同步节点写直接抛 `ValueError`，且必须走 `asyncio.run(ainvoke(...))`；
10. **默认 RetryPolicy 不重试 ValueError 这类业务异常**（特性不是 bug），想重试要显式 `retry_on` 声明。

## 官方链接

- 持久化与时间旅行：<https://docs.langchain.com/oss/python/langgraph/persistence>
- 子图：<https://docs.langchain.com/oss/python/langgraph/use-subgraphs>
- 子图持久化（use-subgraphs 内 "Subgraph persistence"）：<https://docs.langchain.com/oss/python/langgraph/use-subgraphs#subgraph-persistence>
- 容错：<https://docs.langchain.com/oss/python/langgraph/fault-tolerance>
- 测试：<https://docs.langchain.com/oss/python/langgraph/testing>
- 图 API（RetryPolicy / 超时等节点参数）：<https://docs.langchain.com/oss/python/langgraph/graph-api>